# EXO_SENTINEL — Audit & Diagnostic Manuel
```
╔══════════════════════════════════════════════════════════╗
║  SENTINEL v2.0 — Notebook d'audit autonome              ║
║  Usage : diagnostic approfondi, relecture ledger,       ║
║          test SENTINEL hors workflow frégate            ║
╚══════════════════════════════════════════════════════════╝
```

**Doctrine :** SENTINEL prépare le contexte. Vulkan prescrit. L'Empereur valide.

---
## Quand utiliser ce notebook
| Cas | Action ici |
|-----|------------|
| Diagnostiquer une frégate manuellement | Section 2 |
| Relire / rechercher dans le ledger | Section 3 |
| Simuler une erreur et tester SENTINEL | Section 4 |
| Analyser un dossier de frames existant | Section 5 |
| Générer un rapport complet multi-frégate | Section 6 |

## 1. Setup

In [ ]:
# ── Configuration — adapter si besoin ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

# Chemin racine EXODUS sur Drive
DRIVE_ROOT = Path('/content/drive/MyDrive/EXODUS_V2')
SENTINEL_BASE = DRIVE_ROOT / 'SENTINEL_CORE'

# Import SENTINEL
sys.path.insert(0, str(SENTINEL_BASE / 'CODEBASE'))
from sentinel_core import Sentinel

sentinel = Sentinel(base_dir=str(SENTINEL_BASE))
print('[SENTINEL] Prêt.')
print(f'  Base dir : {SENTINEL_BASE}')
print(f'  Ledger   : {SENTINEL_BASE / "memory.json"}')


## 2. Audit Manuel — Lancer SENTINEL sur une frégate

In [ ]:
# ── Configurer ici ──────────────────────────────────────────────────────────
FREGATE = 'U03'               # U00 à U06
MODE    = 'blend'              # 'blend' (Colab+Blender) | 'frames' (local/Colab)

# Chemin selon le mode
BLEND_PATH  = str(DRIVE_ROOT / '03_SCENOGRAPHY_DOCK/OUT_PREMIUM_SCENE/environment_1.blend')
FRAMES_PATH = str(DRIVE_ROOT / '04_PHOTOGRAPHY_WING/OUT_CAMERA_FRAMES')

# ── Lancer ───────────────────────────────────────────────────────────────────
if MODE == 'blend':
    rapport = sentinel.run(fregate=FREGATE, blend_path=BLEND_PATH)
else:
    rapport = sentinel.run(fregate=FREGATE, frames_dir=FRAMES_PATH)

# ── Afficher résultats ────────────────────────────────────────────────────────
print(f'\n[SENTINEL] Verdict : {rapport["verdict"]}')
print(f'  Frégate           : {rapport["fregate"]}')
print(f'  Timestamp         : {rapport["timestamp"]}')
print(f'  Injections ledger : {rapport["ledger_injections"]}')
print(f'  Fichiers sauvegardés : {rapport["fichiers_sauvegardes"]}')

if rapport.get('state_sig'):
    sig = rapport['state_sig']
    print(f'\n[B2 State] verdict={sig.get("verdict")} | erreurs={sig.get("erreurs",[])}')

if rapport.get('diagnostic'):
    diag = rapport['diagnostic']
    print(f'[B5 Diag]  conclusion={diag.get("conclusion")} | action={diag.get("action","")[:80]}')


In [ ]:
# ── Prompt Vulkan (si FAIL) ──────────────────────────────────────────────────
if rapport['verdict'] == 'FAIL':
    print('\n' + '='*60)
    print('  PROMPT VULKAN — Copier dans Claude')
    print('='*60)
    print(rapport.get('prompt_vulkan', 'Aucun prompt généré'))
else:
    print(f'[SENTINEL] {rapport["verdict"]} — Aucune action requise.')


## 3. Ledger — Relecture de la mémoire

In [ ]:
from brique6_ledger import Ledger
import json

ledger = Ledger(memory_path=str(SENTINEL_BASE / 'memory.json'))

# Toutes les entrées
all_entries = ledger.list_entries()
print(f'[LEDGER] {len(all_entries)} entrée(s) au total\n')

for e in all_entries:
    print(f"  [{e.get('fregate','?')}] {e.get('timestamp','?')[:19]} | erreur: {str(e.get('erreur',''))[:60]}")


In [ ]:
# ── Filtrer par frégate ──────────────────────────────────────────────────────
FREGATE_FILTER = 'U03'  # Changer ici

entries = ledger.get_injections(fregate=FREGATE_FILTER)
print(f'[LEDGER] {len(entries)} entrée(s) pour {FREGATE_FILTER}')
for e in entries:
    print(f"  Erreur     : {e.get('erreur','')}")
    print(f"  Cause      : {e.get('cause','')}")
    print(f"  Correction : {e.get('correction','')}")
    print()


## 4. Simulation d'erreur — Tester SENTINEL

In [ ]:
# ── Test : SENTINEL doit détecter manuellement un état FAIL ─────────────────
# On passe un chemin vide pour forcer une erreur B2
print('[TEST] Simulation — blend path invalide')

test_rapport = sentinel.run(
    fregate='U03',
    blend_path='/tmp/fake_scene_inexistante.blend'
)

print(f'Verdict attendu : FAIL | Obtenu : {test_rapport["verdict"]}')
assert test_rapport['verdict'] in ('FAIL', 'WARN'), 'SENTINEL aurait dû détecter le problème'
print('[TEST] SENTINEL réagit correctement aux erreurs.')


## 5. Analyse d'un dossier de frames

In [ ]:
from brique3_ghost import GhostRenderer

# ── Configurer ────────────────────────────────────────────────────────────────
FRAMES_DIR = str(DRIVE_ROOT / '04_PHOTOGRAPHY_WING/OUT_CAMERA_FRAMES')

ghost = GhostRenderer()
result = ghost.analyze_folder(FRAMES_DIR)
ghost.print_report(result)

print(f"\nTotal frames : {result.get('frames_total', 0)}")
print(f"Échantillon analysé : {result.get('frames_sampled', 0)}")
if result.get('sample_results'):
    for r in result['sample_results']:
        icon = 'OK' if r['verdict'] == 'VISIBLE' else '!!'
        print(f"  [{icon}] {r['frame']:<40} luma={r['luminance']}")


## 6. Rapport multi-frégate (audit global)

In [ ]:
# ── Auditer toutes les frégate qui ont des frames disponibles ────────────────
FRIGATES_FRAMES = {
    'U04': str(DRIVE_ROOT / '04_PHOTOGRAPHY_WING/OUT_CAMERA_FRAMES'),
    'U05': str(DRIVE_ROOT / '05_ALCHEMIST_LAB/OUT_FINAL_FRAMES'),
}

resultats = {}
for fregate, frames_path in FRIGATES_FRAMES.items():
    print(f'\n[SENTINEL] Audit {fregate}...')
    r = sentinel.run(fregate=fregate, frames_dir=frames_path)
    resultats[fregate] = r['verdict']
    print(f'  → {r["verdict"]}')

print('\n' + '='*50)
print('  RAPPORT GLOBAL SENTINEL')
print('='*50)
for fregate, verdict in resultats.items():
    icon = 'PASS' if verdict == 'PASS' else ('WARN' if verdict == 'WARN' else 'FAIL')
    print(f'  [{icon}] {fregate} : {verdict}')


---
*SENTINEL v2.0 — Notebook d'audit autonome*  
*SENTINEL veille. L'Empire est immortel.*